In [1]:
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5-20251001"

import json



In [2]:
def add_user_message(messages, content):
    messages.append({
        "role": "user",
        "content": content
    })


def add_assistant_message(messages, content):
    messages.append({
        "role": "assistant",
        "content": content
    })

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, '```json')
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)


In [3]:
dataset = generate_dataset()

'''
[{'task': "Write a Python function that parses an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file') and returns just the bucket name."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'company-data'."},
 {'task': 'Write a regular expression that matches and validates AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters).'}]
 '''

'\n[{\'task\': "Write a Python function that parses an AWS S3 bucket name from an S3 URI (e.g., \'s3://my-bucket/path/to/file\') and returns just the bucket name."},\n {\'task\': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named \'company-data\'."},\n {\'task\': \'Write a regular expression that matches and validates AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters).\'}]\n '

In [ ]:
## Now start evaluation

def run_prompt(test_case): 
    '''
    Merges the prompt and the test case, sends it to the model, and returns the model's response.
    '''

    prompt = f'''
    Please solve the following task: {test_case['task']}.
'''
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

##

def run_test_case (test_case):
    '''
    calls run_prompt, then grade the ouput '''

    output = run_prompt(test_case)

    score = 10 

    return 

def run_eval (dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

    

In [6]:
with open ("../dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

print(f"result is {result}")


result is [{'output': '# AWS S3 Bucket Name Parser\n\nHere\'s a Python function that parses an AWS S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    """\n    Parse the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): An S3 URI in the format \'s3://bucket-name/path/to/file\'\n        \n    Returns:\n        str: The S3 bucket name\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n        \n    Examples:\n        >>> parse_s3_bucket_name(\'s3://my-bucket/path/to/file\')\n        \'my-bucket\'\n        >>> parse_s3_bucket_name(\'s3://another-bucket/\')\n        \'another-bucket\'\n        >>> parse_s3_bucket_name(\'s3://bucket-name\')\n        \'bucket-name\'\n    """\n    if not s3_uri:\n        raise ValueError("S3 URI cannot be empty")\n    \n    # Check if the URI starts with \'s3://\'\n    if not s3_uri.startswith(\'s3://\'):\n        raise ValueError("Invalid S3 URI: must start with \'s3://\

In [7]:
type (result)

list

result is [{'output': '# AWS S3 Bucket Name Parser\n\nHere\'s a Python function that parses an AWS S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    """\n    Parse the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): An S3 URI in the format \'s3://bucket-name/path/to/file\'\n        \n    Returns:\n        str: The S3 bucket name\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n        \n    Examples:\n        >>> parse_s3_bucket_name(\'s3://my-bucket/path/to/file\')\n        \'my-bucket\'\n        >>> parse_s3_bucket_name(\'s3://another-bucket/\')\n        \'another-bucket\'\n        >>> parse_s3_bucket_name(\'s3://bucket-name\')\n        \'bucket-name\'\n    """\n    if not s3_uri:\n        raise ValueError("S3 URI cannot be empty")\n    \n    # Check if the URI starts with \'s3://\'\n    if not s3_uri.startswith(\'s3://\'):\n        raise ValueError("Invalid S3 URI: must start with \'s3://\'")\n    \n    # Remove the \'s3://\' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Extract the bucket name (everything before the first \'/\')\n    bucket_name = uri_without_prefix.split(\'/\')[0]\n    \n    # Validate that bucket name is not empty\n    if not bucket_name:\n        raise ValueError("Invalid S3 URI: bucket name is empty")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == "__main__":\n    # Test cases\n    test_cases = [\n        (\'s3://my-bucket/path/to/file\', \'my-bucket\'),\n        (\'s3://another-bucket/\', \'another-bucket\'),\n        (\'s3://bucket-name\', \'bucket-name\'),\n        (\'s3://my-bucket/folder/subfolder/file.txt\', \'my-bucket\'),\n        (\'s3://test-123/a/b/c/d/e.json\', \'test-123\'),\n    ]\n    \n    print("Testing parse_s3_bucket_name():\\n")\n    for uri, expected in test_cases:\n        result = parse_s3_bucket_name(uri)\n        status = "✓" if result == expected else "✗"\n        print(f"{status} Input: {uri}")\n        print(f"  Expected: {expected}")\n        print(f"  Got:      {result}\\n")\n    \n    # Test error cases\n    print("\\nTesting error cases:\\n")\n    error_cases = [\n        (\'\', "S3 URI cannot be empty"),\n        (\'http://my-bucket/path\', "must start with \'s3://\'"),\n        (\'s3://\', "bucket name is empty"),\n    ]\n    \n    for uri, expected_error in error_cases:\n        try:\n            parse_s3_bucket_name(uri)\n            print(f"✗ Expected error for: {uri}")\n        except ValueError as e:\n            print(f"✓ Input: {uri}")\n            print(f"  Error: {e}\\n")\n```\n\n## Alternative Using Regular Expressions\n\nIf you prefer a regex-based approach:\n\n```python\nimport re\n\ndef parse_s3_bucket_name_regex(s3_uri: str) -> str:\n    """\n    Parse the bucket name from an S3 URI using regex.\n    \n    Args:\n        s3_uri (str): An S3 URI in the format \'s3://bucket-name/path/to/file\'\n        \n    Returns:\n        str: The S3 bucket name\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n    """\n    match = re.match(r\'s3://([a-zA-Z0-9.\\-_]+)(?:/|$)\', s3_uri)\n    ', 'test_case': {'task': "Write a Python function that parses an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file') and returns just the bucket name."}, 'score': 10}, {'output': '# AWS IAM Read-Only S3 Bucket Policy\n\n```json\n{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Sid": "ListBucketContents",\n      "Effect": "Allow",\n      "Action": [\n        "s3:ListBucket",\n        "s3:GetBucketVersioning",\n        "s3:GetBucketLocation"\n      ],\n      "Resource": "arn:aws:s3:::company-data"\n    },\n    {\n      "Sid": "ReadObjectsInBucket",\n      "Effect": "Allow",\n      "Action": [\n        "s3:GetObject",\n        "s3:GetObjectVersion",\n        "s3:GetObjectTagging"\n      ],\n      "Resource": "arn:aws:s3:::company-data/*"\n    }\n  ]\n}\n```\n\n## Policy Explanation\n\n| Component | Description |\n|-----------|-------------|\n| **ListBucketContents** | Allows users to list and view bucket properties |\n| **ReadObjectsInBucket** | Allows users to retrieve/download objects from the bucket |\n| **Resource ARNs** | `arn:aws:s3:::company-data` for bucket-level operations and `arn:aws:s3:::company-data/*` for object-level operations |\n\n## Included Permissions\n\n- ✅ List bucket contents\n- ✅ Download/Get objects\n- ✅ View object versions and tags\n- ✅ View bucket location and versioning settings\n- ❌ Cannot upload, delete, or modify objects\n\nThis policy provides complete read-only access to the \'company-data\' bucket while preventing any write operations.', 'test_case': {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'company-data'."}, 'score': 10}, {'output': '# AWS EC2 Instance ID Regex\n\nHere\'s a regular expression that validates AWS EC2 instance IDs:\n\n```regex\n^i-[0-9a-f]{17}$\n```\n\n## Explanation\n\n- `^` - Start of string anchor\n- `i-` - Literal prefix required for all EC2 instance IDs\n- `[0-9a-f]` - Character class matching hexadecimal digits (0-9 and a-f)\n- `{17}` - Exactly 17 hexadecimal characters\n- `$` - End of string anchor\n\n## Examples\n\n### Valid Matches\n```\ni-0a1b2c3d4e5f6g7h8\ni-1234567890abcdef0\ni-0000000000000000a\n```\n\n### Invalid Matches\n```\ni-0a1b2c3d4e5f6g7h        (too short - 16 chars)\ni-0a1b2c3d4e5f6g7h89      (too long - 18 chars)\ni-0a1b2c3d4e5f6g7H        (uppercase hex not allowed)\n0a1b2c3d4e5f6g7h8         (missing \'i-\' prefix)\n```\n\n## Case-Insensitive Variant\n\nIf you need to match uppercase hexadecimal characters as well:\n\n```regex\n^i-[0-9a-fA-F]{17}$\n```\n\n## Code Examples\n\n### JavaScript\n```javascript\nconst ec2Regex = /^i-[0-9a-f]{17}$/;\nconsole.log(ec2Regex.test(\'i-0a1b2c3d4e5f6g7h8\')); // true\n```\n\n### Python\n```python\nimport re\nec2_pattern = r\'^i-[0-9a-f]{17}$\'\nre.match(ec2_pattern, \'i-0a1b2c3d4e5f6g7h8\')  # Match object\n```\n\n### Go\n```go\npackage main\nimport "regexp"\npattern := `^i-[0-9a-f]{17}$`\nre := regexp.MustCompile(pattern)\nre.MatchString("i-0a1b2c3d4e5f6g7h8") // true\n```', 'test_case': {'task': 'Write a regular expression that matches and validates AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters).'}, 'score': 10}, {'output': '# AWS S3 Bucket Name Validator\n\nHere\'s a comprehensive solution with validation, tests, and documentation:\n\n```python\nimport re\nfrom typing import Tuple\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    """\n    Validates if a given string is a valid AWS S3 bucket name.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters (a-z), numbers (0-9), and hyphens (-)\n    - Cannot start or end with a hyphen\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The bucket name to validate\n        \n    Returns:\n        bool: True if valid, False otherwise\n    """\n    # Check if input is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Rule 1: Check length (3-63 characters)\n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    # Rule 2: Check if it contains only lowercase letters, numbers, and hyphens\n    if not re.match(r\'^[a-z0-9-]+$\', bucket_name):\n        return False\n    \n    # Rule 3: Cannot start or end with a hyphen\n    if bucket_name.startswith(\'-\') or bucket_name.endswith(\'-\'):\n        return False\n    \n    # Rule 4: Cannot be formatted as an IP address\n    if re.match(r\'^(\\d{1,3}\\.){3}\\d{1,3}$\', bucket_name):\n        return False\n    \n    return True\n\n\ndef validate_s3_bucket_name(bucket_name: str) -> Tuple[bool, str]:\n    """\n    Validates an S3 bucket name and returns detailed feedback.\n    \n    Args:\n        bucket_name (str): The bucket name to validate\n        \n    Returns:\n        Tuple[bool, str]: (is_valid, message)\n    """\n    if not isinstance(bucket_name, str):\n        return False, "Bucket name must be a string"\n    \n    if len(bucket_name) < 3:\n        return False, "Bucket name must be at least 3 characters long"\n    \n    if len(bucket_name) > 63:\n        return False, "Bucket name cannot exceed 63 characters"\n    \n    if not re.match(r\'^[a-z0-9-]+$\', bucket_name):\n        return False, "Bucket name can only contain lowercase letters, numbers, and hyphens"\n    \n    if bucket_name.startswith(\'-\'):\n        return False, "Bucket name cannot start with a hyphen"\n    \n    if bucket_name.endswith(\'-\'):\n        return False, "Bucket name cannot end with a hyphen"\n    \n    if re.match(r\'^(\\d{1,3}\\.){3}\\d{1,3}$\', bucket_name):\n        return False, "Bucket name cannot be formatted as an IP address"\n    \n    return True, "Bucket name is valid"\n\n\n# Test cases\ndef run_tests():\n    """Run comprehensive tests for the validator"""\n    \n    test_cases = [\n        # (bucket_name, expected_result, description)\n        ("my-bucket", True, "Valid: lowercase with hyphens"),\n        ("my-bucket-123", True, "Valid: lowercase, numbers, and hyphens"),\n        ("bucket123", True, "Valid: lowercase and numbers"),\n        ("a1b", True, "Valid: minimum length (3 chars)"),\n        ("a" * 63, True, "Valid: maximum length (63 chars)"),\n        \n        # Invalid cases\n        ("My-Bucket", False, "Invalid: contains uppercase letters"),\n        ("my_bucket", False, "Invalid: contains underscore"),\n        ("my bucket", False, "Invalid: contains space"),\n        ("-mybucket", False, "Invalid: starts with hyphen"),\n        ("mybucket-", False, "Invalid:', 'test_case': {'task': 'Write a Python function that validates if a given string is a valid AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens only, cannot start or end with hyphen)'}, 'score': 10}, {'output': '# AWS IAM Read-Only S3 Bucket Policy\n\n```json\n{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Sid": "ListBucketContents",\n      "Effect": "Allow",\n      "Action": [\n        "s3:GetObject",\n        "s3:GetObjectVersion"\n      ],\n      "Resource": "arn:aws:s3:::my-data-bucket/*"\n    },\n    {\n      "Sid": "ListBucket",\n      "Effect": "Allow",\n      "Action": [\n        "s3:ListBucket",\n        "s3:GetBucketVersioning"\n      ],\n      "Resource": "arn:aws:s3:::my-data-bucket"\n    }\n  ]\n}\n```\n\n## Explanation\n\nThis policy grants the following read-only permissions:\n\n| Action | Purpose |\n|--------|---------|\n| `s3:GetObject` | Read objects from the bucket |\n| `s3:GetObjectVersion` | Read specific object versions |\n| `s3:ListBucket` | List bucket contents |\n| `s3:GetBucketVersioning` | Check bucket versioning status |\n\n**Resource ARNs:**\n- `arn:aws:s3:::my-data-bucket/*` - For object-level operations\n- `arn:aws:s3:::my-data-bucket` - For bucket-level operations\n\nThis policy **does NOT** allow:\n- Writing or uploading objects\n- Deleting objects\n- Modifying bucket configuration\n- Any administrative actions', 'test_case': {'task': "Write a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'"}, 'score': 10}, {'output': '# AWS ARN Region Code Extraction\n\nHere\'s a regular expression solution to extract the region code from an AWS ARN:\n\n## Solution\n\n```python\nimport re\n\ndef extract_region_from_arn(arn):\n    """Extract region code from AWS ARN string"""\n    pattern = r\'arn:aws:[^:]+:([^:]*):?\'\n    match = re.search(pattern, arn)\n    return match.group(1) if match else None\n\n# Test cases\ntest_arns = [\n    \'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket\',\n    \'arn:aws:ec2:eu-west-1:123456789012:instance/i-1234567890abcdef0\',\n    \'arn:aws:iam::123456789012:user/Development/product_1234/*\',\n    \'arn:aws:lambda:ap-southeast-1:123456789012:function:my-function\',\n    \'arn:aws:s3:::my-bucket\',  # S3 has no region\n]\n\nfor arn in test_arns:\n    region = extract_region_from_arn(arn)\n    print(f"{arn}")\n    print(f"  → Region: \'{region}\'\\n")\n```\n\n## Output\n\n```\narn:aws:s3:us-east-1:123456789012:bucket/my-bucket\n  → Region: \'us-east-1\'\n\narn:aws:ec2:eu-west-1:123456789012:instance/i-1234567890abcdef0\n  → Region: \'eu-west-1\'\n\narn:aws:iam::123456789012:user/Development/product_1234/*\n  → Region: \'\'\n\narn:aws:lambda:ap-southeast-1:123456789012:function:my-function\n  → Region: \'ap-southeast-1\'\n\narn:aws:s3:::my-bucket\n  → Region: \'\'\n```\n\n## Regex Explanation\n\n| Pattern | Meaning |\n|---------|---------|\n| `arn:aws:` | Literal prefix |\n| `[^:]+` | Service name (one or more non-colon chars) |\n| `:` | Separator |\n| `([^:]*)` | **Capturing group**: region code (zero or more non-colon chars) |\n| `:?` | Optional colon after region |\n\n## Key Points\n\n- **Empty region** is valid (e.g., IAM, S3 ARNs return empty string `\'\'`)\n- **Non-greedy matching** using `[^:]*` stops at the next colon\n- **Flexible design** handles all AWS service ARNs\n- Returns `None` if the ARN doesn\'t match the expected format\n\n## Alternative: More Strict Version\n\nIf you want to ensure a valid region pattern (letters, numbers, hyphens):\n\n```python\npattern = r\'arn:aws:[^:]+:([a-z0-9\\-]*):?\'\n```\n\nThis restricts regions to valid AWS region naming conventions.', 'test_case': {'task': "Write a regular expression that extracts the region code from an AWS ARN string (e.g., 'us-east-1' from 'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket')"}, 'score': 10}]